# caban — pipeline driver

Notebook entry point for the caban analysis pipeline. Each cell runs one stage; you can stop at any point and inspect state in the live kernel.

**Architecture**
- `caban.config.PipelineConfig` — all user-tunable switches.
- `caban.loader.load_all_mice(cfg)` — returns a `SimpleNamespace` (`ds`) with every metadata dict, session dict, mapping accumulator, and engram-pass result.
- `caban.sections` — one `run_<name>(ds, cfg, ...)` function per top-level analysis. Each cell below calls one.
- `caban.pipeline.*` — thin wrappers around the heavier analysis modules (`caban.population`, `caban.isomap`, `caban.epoch_analysis`, ...).

**Live debugging.** To iterate on a section's body, open `caban/sections.py`, find the corresponding `run_<name>` function, copy its body (including the ds/cfg unpacking block) into a scratch cell, and run it inline. `ds`, `cfg`, and any upstream cross-section state are already in scope.


## 1. Setup

    "`%autoreload 2` rebuilds modules on edit. Heavy code lives in `caban/*.py` and is reloaded automatically; the dataset itself is **not** rebuilt — you keep your loaded `ds`."

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, importlib
import numpy as np
from IPython import get_ipython

ENABLE_PLOTTING = True  # Flip to False for headless/high-throughput runs.

import matplotlib
if ENABLE_PLOTTING:
    # Keep notebook rendering active when plotting is enabled.
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("matplotlib", "inline")
else:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
if not ENABLE_PLOTTING:
    plt.ioff()

import caban.config
import caban.loader
import caban.pipeline
from caban.config import PipelineConfig
from caban.loader import load_all_mice
from caban.pipeline import (
    resolve_continuity_params,
    engram_idx_by_mouse,
    run_engram_sanity_plots,
    run_population_pca,
    run_population_pca_all_modes,
    run_isomap,
    run_epoch_pv,
    run_cross_session_epoch_pv,
)

## 2. Build config

Edit fields here to override defaults. All ~90 switches from `caban/main.py` are exposed.


In [ ]:
cfg = PipelineConfig(
    DEVEL_SWITCH=False,
    PLOTS_DIR_SINGULAR=True,
    # DEBUG=True,                # restrict to one mouse for quick smoke tests
    # plot_pf_raw_maps=True,     # heavy per-cell PF plots
    # optimize_parameters=True,  # run raw-S decoder Optuna study
    # enable_population_curve=True,
)
print(cfg)

## 3. Load all mice (with optional pickle cache)

`load_all_mice(cfg)` builds CrossReg + Session objects, runs the per-mapping accumulators, the unified engram-identity pass, and the merged-PF backfill. This is the slow step.

`load_all_mice` handles caching internally:
- If `<NPY_SAVE_PATH>/ds_cache.pkl` exists → loads `ds` from pickle and skips the fresh build.
- Otherwise → builds `ds` fresh and writes the pickle for next time.
- Always prints a deep-size memory breakdown after load.

Pass `use_cache=False` to force a rebuild without touching the pickle; pass `cache_path=...` to override the default location; pass `report_memory=False` to suppress the size breakdown.


In [ ]:
ds = load_all_mice(cfg)

# 6.0s

In [ ]:
# Dump per-mouse behaviour timing parameters (miniscope-frame exp bounds,
# tone/shock onsets/offsets in seconds relative to both experiment start and
# raw recording start). Writes <cfg.PLOTS_DIR>/behaviour_params.py / .m.
from caban.sections import dump_behaviour_params
dump_behaviour_params(ds, cfg)

In [ ]:
#globals().update(vars(ds))

## 3. Load analysis modules

Here we load the analysis routines that will be performed in the "Analysis sections" area. 

In [ ]:
# Section functions live in caban.sections.
# Each takes (ds, cfg) plus any cross-section state via kwargs
# and is gated by its cfg.plot_* / enable_* switch.
from caban.sections import (
    run_want_sample_traces_paper,
    run_sp_rates,
    run_binned_sp_rates,
    run_ROIs,
    run_proportional_activities,
    run_LT_firing_rate_changes,
    run_PSTH,
    run_pf_and_loc,
    run_occupancy_analysis,
    run_LT_pfs,
    run_LT_decoding,
    run_zone_crossreg,
    run_continuity_and_paramsets,
    run_optimize_raw_decoder,
    run_optimize_pf_decoder,
    run_paradigm_A,
    run_paradigm_B,
    run_paradigm_C,
    run_paradigm_D1,
    run_paradigm_D2,
    run_paradigm_E1,
    run_paradigm_E2,
    run_paradigm_F,
    run_mixedlm_cross_vs_within,
    run_mixedlm_vs_tfc_cond,
    run_pv_correlation_2d,
    run_epoch_pv_within,
    run_epoch_pv_cross,
    run_population_pca,
    run_isomap,
    run_umap,
    run_crossreg_pca_umap,
    run_population_vectors,
    run_population_vector_distances,
    run_binned_activities,
    run_avg_population_activity,
    run_rastermap_single_mouse,
    run_rastermap_sweep,
 )

# Shared state threaded between sections (None until produced).
raw_params = pf_params = None
lt_cont_pvt = None; use_PCT_error = None
mt_A_results_2D = mt_B_results_2D = mt_C_results_2D = None

# Analysis sections

The cells below follow the exact top-level pathway of `caban/main.py`, one section per cell, in the same order. Each cell calls a `run_<name>(ds, cfg, ...)` function from `caban.sections`.

Every block is gated by its own `cfg.plot_*` / `cfg.enable_*` / `cfg.optimize_*` switch. Toggle on `cfg` and re-run the cell — no globals re-sync needed.


### 8. Sample traces

Plot sample traces for paper.

In [ ]:
import importlib
import caban.analysis
import caban.sections

importlib.reload(caban.analysis)
importlib.reload(caban.sections)

In [ ]:
run_want_sample_traces_paper(ds, cfg, selection_mode=True, len_trace=1200)


### 9. Spike-rate panels

`caban/main.py` L1474–1539


In [ ]:
run_sp_rates(ds, cfg)


### 10. Binned spike-rate panels

`caban/main.py` L1540–1590


In [ ]:
run_binned_sp_rates(ds, cfg)


In [ ]:
globals().update(vars(ds))

In [ ]:

m='G05'
sess=TFC_cond[m]
#cell_ = 20

import random
num_random_cells = 10
plt.figure(); 
#plt.plot(sess.S[cell_,:], 'r', alpha=0.5); 
random_cells = random.sample(range(sess.S.shape[0]), num_random_cells)
for cell in random_cells:
    plt.plot(sess.S[cell, :], 'r', alpha=0.5)
plt.plot(sess.velocities_miniscope, 'b'); 
plt.plot(sess.velocities_miniscope_smooth, 'k')


def plot_random_cells(mouse, session, num_random_cells=10, multiplier=1):
    """
    Plots random cells' activity for a given mouse and session.
    Parameters:
    mouse (str): Identifier for the mouse.
    session (dict): Dictionary containing session data for each mouse.
    num_random_cells (int, optional): Number of random cells to plot. Default is 10.
    The function selects a random subset of cells from the session data and plots their activity.
    It also overlays velocity data and event markers (tone and shock onsets/offsets) on the plots.
    Example usage:
    --------------
    plot_random_cells('mouse1', session_data, num_random_cells=10)
    """

    sess = session[mouse]
    random_cells = random.sample(range(sess.S.shape[0]), num_random_cells)
    fig, axes = plt.subplots(3, 3, figsize=(15, 15), sharex=True, sharey=True)
    fig.suptitle(f'Random Cells for Mouse {mouse}', fontsize=16)
    for i, cell in enumerate(random_cells[:9]):
        ax = axes[i // 3, i % 3]
        ax.plot(sess.velocities_miniscope, 'b', alpha=0.5)
        ax.plot(sess.velocities_miniscope_smooth, 'k', alpha=0.5)
        for x in sess.tone_onsets * multiplier:
            ax.axvline(x, c='b', ls='--')
        for x in sess.tone_offsets * multiplier:
            ax.axvline(x, c='b', ls='--')
        for x in sess.shock_onsets * multiplier:
            ax.axvline(x, c='r', ls='--')
        for x in sess.shock_offsets * multiplier:
            ax.axvline(x, c='r', ls='--')
        norm_value = sess.S[cell,:].max()
        ax.plot(sess.S[cell, :], 'r') 
        ax.plot((sess.C[cell, :]/sess.C[cell,:].max())*norm_value, 'orange') 
        ax.text(0.95, 0.95, f'Cell {cell}', transform=ax.transAxes, fontsize=12,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(facecolor='white', alpha=1))
    plt.tight_layout()
    print(random_cells[:9])

# Example usage
m='G05'
sess=TFC_cond
plot_random_cells(m, TFC_cond)

### 11. ROI maps

`caban/main.py` L1591–1598


In [ ]:
run_ROIs(ds, cfg)


### 12. Proportional activities

`caban/main.py` L1599–1609


In [ ]:
run_proportional_activities(ds, cfg)


### 13. LT firing-rate changes

`caban/main.py` L1610–1646


In [ ]:
run_LT_firing_rate_changes(ds, cfg)


### 14. PSTH

`caban/main.py` L1647–1681


In [ ]:
run_PSTH(ds, cfg)


### 15. Place fields and locations

`caban/main.py` L1682–1779


In [ ]:
run_pf_and_loc(ds, cfg)
# 5m 54.0s

### 16. Occupancy / trajectory / immobility

`caban/main.py` L1780–1818


In [ ]:
run_occupancy_analysis(ds, cfg)
# 1m 47.9s

### 17. LT place fields

`caban/main.py` L1819–2095


In [ ]:
# Temporary insurance to prevent fm loading from pickles crashing since they don't know about SSTCa2->caban rename.
import sys
import caban.spatial
sys.modules['SSTCa2_spatial'] = caban.spatial

In [ ]:
run_LT_pfs(ds, cfg)
# 59m 4.6s

In [ ]:
from caban.loader import report_ds_memory
report_ds_memory(ds)

### 18. LT decoding

`caban/main.py` L2096–2888


In [ ]:
_result = run_LT_decoding(ds, cfg)
lt_cont_pvt   = _result.get("lt_cont_pvt") if _result else None
use_PCT_error = _result.get("use_PCT_error") if _result else None


### 19. Zone cross-registration suite

`caban/main.py` L2889–3037


In [ ]:
run_zone_crossreg(ds, cfg, lt_cont_pvt=lt_cont_pvt, use_PCT_error=use_PCT_error)

# 6m 36.7s

### 20. Decoder constants and continuity sigmas

`caban/main.py` L3038–3215


In [ ]:
_result = run_continuity_and_paramsets(ds, cfg)
raw_params = _result["raw_params"] if _result else None
pf_params  = _result["pf_params"]  if _result else None

# 28.3s 

### 21. Optimize raw-decoder parameters

`caban/main.py` L3216–3285


In [ ]:
run_optimize_raw_decoder(ds, cfg)
# normally should not run as optimize_parameters=False

### 22. Optimize PF-decoder parameters

`caban/main.py` L3286–3537


In [ ]:
run_optimize_pf_decoder(ds, cfg)
# normally should not run as optimize_parameters=False

### 23. 2D Bayesian decoder paradigm A

`caban/main.py` L3783–4016


In [ ]:
_result = run_paradigm_A(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_A_results_2D = _result["mt_A_results_2D"] if _result else None

# 18m 32.7s

### 24. 2D Bayesian decoder paradigm B

`caban/main.py` L4017–4171


In [ ]:
_result = run_paradigm_B(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_B_results_2D = _result["mt_B_results_2D"] if _result else None

# 15m 14.0s

### 25. 2D Bayesian decoder paradigm C

`caban/main.py` L4172–4327


In [ ]:
_result = run_paradigm_C(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_C_results_2D = _result["mt_C_results_2D"] if _result else None

# 15m 25.3s

### 26. 2D Bayesian decoder paradigm D1

`caban/main.py` L4328–4477


In [ ]:
run_paradigm_D1(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 11m 4.8s

### 27. 2D Bayesian decoder paradigm D2

`caban/main.py` L4478–4627


In [ ]:
run_paradigm_D2(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 47.7s

### 28. 2D Bayesian decoder paradigm E1

`caban/main.py` L4628–4775


In [ ]:
run_paradigm_E1(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 58.2s

### 29. 2D Bayesian decoder paradigm E2

`caban/main.py` L4776–4924


In [ ]:
run_paradigm_E2(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 38.1s

### 30. 2D Bayesian decoder paradigm F

`caban/main.py` L4925–5127


In [ ]:
run_paradigm_F(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 21.8s

### 31. TFC cross-session vs within-session MixedLM

`caban/main.py` L5128–5155


In [ ]:
run_mixedlm_cross_vs_within(ds, cfg, mt_A_results_2D=mt_A_results_2D, mt_B_results_2D=mt_B_results_2D, mt_C_results_2D=mt_C_results_2D)

# 2m 45.7s

### 32. TFC vs TFC_cond baseline MixedLM

`caban/main.py` L5156–5181


In [ ]:
run_mixedlm_vs_tfc_cond(ds, cfg, mt_A_results_2D=mt_A_results_2D, mt_B_results_2D=mt_B_results_2D, mt_C_results_2D=mt_C_results_2D)

# 2m 39.4s

### 33. 2D Population Vector (PV) correlation

`caban/main.py` L5182–5562


In [ ]:
run_pv_correlation_2d(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 70m 47.0s

### 34. Within-session epoch-PV similarity

`caban/main.py` L5563–5631


In [ ]:
run_epoch_pv_within(ds, cfg)

# 17m 14.1s

### 35. Cross-session epoch-PV similarity

`caban/main.py` L5632–5708


In [ ]:
run_epoch_pv_cross(ds, cfg)

# >44m

### 36. Population PCA trajectory analysis (incl. engram sanity)

`caban/main.py` L5709–5861


In [ ]:
run_population_pca(ds, cfg)

# 113m 51.6s

In [ ]:
mapping_tfc_b_b1wk = (
    ds.mapping_TFC_cond_Test_B_Test_B_1wk
    if hasattr(ds, 'mapping_TFC_cond_Test_B_Test_B_1wk')
    else 'TFC_cond+Test_B+Test_B_1wk'
)

crossreg_by_mouse = ds.TFC_B_B_1wk_crossreg

mice = sorted(set(ds.TFC_cond) & set(ds.Test_B) & set(ds.Test_B_1wk))
if not mice:
    raise RuntimeError('No mice found in the intersection of TFC_cond, Test_B, and Test_B_1wk.')

rows = []
for mouse in mice:
    if mouse not in crossreg_by_mouse:
        raise RuntimeError(f'Missing crossreg object for mouse {mouse}.')

    n_tfc = ds.TFC_cond[mouse].S.shape[0]
    n_b = ds.Test_B[mouse].S.shape[0]
    n_b1wk = ds.Test_B_1wk[mouse].S.shape[0]

    if n_tfc <= 0:
        raise RuntimeError(f'{mouse}: TFC_cond has zero cells; cannot compute crossreg percentage.')

    df_map = crossreg_by_mouse[mouse].get_mappings_cells(mapping_type=mapping_tfc_b_b1wk)
    n_crossreg = len(df_map)
    pct_crossreg_of_tfc = 100.0 * n_crossreg / n_tfc

    rows.append((mouse, n_tfc, n_b, n_b1wk, n_crossreg, pct_crossreg_of_tfc))

sum_tfc = sum(r[1] for r in rows)
sum_b = sum(r[2] for r in rows)
sum_b1wk = sum(r[3] for r in rows)
sum_crossreg = sum(r[4] for r in rows)
mean_pct = sum(r[5] for r in rows) / len(rows)

if sum_tfc <= 0:
    raise RuntimeError('Total TFC_cond cell count is zero; cannot compute TOTAL percentage.')

pct_total = 100.0 * sum_crossreg / sum_tfc

headers = ('mouse', 'TFC_cond', 'Test_B', 'Test_B_1wk', 'crossreg', '%crossreg_of_TFC_cond')

display_rows = [
    (r[0], str(r[1]), str(r[2]), str(r[3]), str(r[4]), f"{r[5]:.2f}%")
    for r in rows
]

total_row = ('TOTAL', str(sum_tfc), str(sum_b), str(sum_b1wk), str(sum_crossreg), f"{pct_total:.2f}%")
mean_row = (
    'MEAN',
    f"{sum_tfc / len(rows):.2f}",
    f"{sum_b / len(rows):.2f}",
    f"{sum_b1wk / len(rows):.2f}",
    f"{sum_crossreg / len(rows):.2f}",
    f"{mean_pct:.2f}%",
)

widths = [
    max(len(headers[i]), max(len(row[i]) for row in display_rows), len(total_row[i]), len(mean_row[i]))
    for i in range(len(headers))
]

def format_row(values):
    return (
        f"{values[0]:<{widths[0]}}  "
        f"{values[1]:>{widths[1]}}  "
        f"{values[2]:>{widths[2]}}  "
        f"{values[3]:>{widths[3]}}  "
        f"{values[4]:>{widths[4]}}  "
        f"{values[5]:>{widths[5]}}"
    )

line_w = sum(widths) + 2 * (len(widths) - 1)

print(f"Mapping used for crossreg count: {mapping_tfc_b_b1wk}")
print(format_row(headers))
print('-' * line_w)

for row in display_rows:
    print(format_row(row))

print('-' * line_w)
print(format_row(total_row))
print(format_row(mean_row))

### 37. Isomap manifold pipeline

`caban/main.py` L5862–5882


In [ ]:
run_isomap(ds, cfg)

# 213m 26.9s

### 38. UMAP manifold pipeline

`caban/sections.py` run_umap(ds, cfg)

In [ ]:
run_umap(ds, cfg)

# 102m 28.2s

### 39. Cross-registered PCA+UMAP

`caban/sections.py` run_crossreg_pca_umap(ds, cfg)

In [ ]:
# PCA→UMAP cross-registered manifold checks
#
# 1. z-score on TFC_cond, fit PCA/UMAP on TFC_cond
# 2. z-score on Test_B,   fit PCA/UMAP on TFC_cond
# 3. z-score on Test_B,   fit PCA/UMAP on Test_B

umap_pca_results = {}

n_pca_components_l = [5, 10, 20, 30, 50]
umap_n_neighbors_l = [5, 10, 20, 30, 50]

for z_score_sess, fit_sess in [
    ("TFC_cond", "TFC_cond"),
#    ("Test_B", "TFC_cond"), # 7m 41.7s
#    ("Test_B", "Test_B"), # 12m 6.5s
]:
    key = f"zscore_{z_score_sess}__fit_{fit_sess}"

    for n_pca_components in n_pca_components_l:
        for umap_n_neighbors in umap_n_neighbors_l:
            umap_pca_results[key] = run_crossreg_pca_umap(
                ds,
                cfg,
                bin_size_s=0,
                z_score_sess=z_score_sess,
                fit_sess=fit_sess,
                n_pca_components=n_pca_components,
                umap_n_neighbors=umap_n_neighbors,
                umap_min_dist=0.1,
                umap_metric="cosine",
                random_state=None,       # allows parallel UMAP
                umap_n_jobs=-1,          # all cores
                set_parallel_threads=True,
                skip_mice=("G07", "G15"),
                save_embeddings=True,
                auto_close=True,    
            )

# 61m 27.1s

### 40. Cell-averaged population activity

`caban/sections.py` run_avg_population_activity(ds, cfg)

In [ ]:
session_l = ['TFC_cond', 'Test_B', 'Test_B_1wk']
normalize_mode = None  # None | 'per' | 'across'
same_y_across = True
robust_ylim_quantile = None # 0.995  # None disables robust clipping
group_trace_stat = 'mean'  # 'mean' | 'median'
exclude_top_percent_cells = None  # e.g., 5 to remove top 5% most active cells
paper_axes = True
paper_zoom_per_row = True
paper_zoom_quantile = None # 0.995  # set None to use full extrema
paper_row_height = 0.85  # smaller -> tighter vertical stacking
figure_width = 8.0  # width of figures in inches; adjust for aspect ratio
trace_linewidth = 0.5  # line width for all traces
overlay_groups_single_axis = True  # plot hM3D/hM4D/mCherry on one axis
overlay_alpha = 0.65  # transparency for overlay lines
paper_scalebar_time_value = 5  # e.g., 1, 5
paper_scalebar_time_unit = 'min'  # 'min' | 's' | 'frames'
paper_scalebar_amp_value = 0.5
plot_inline = True  # set True to display figures in the notebook
avg_pop_result = run_avg_population_activity(
    ds,
    cfg,
    sessions=session_l,
    normalize=normalize_mode,
    same_y_across=same_y_across,
    robust_ylim_quantile=robust_ylim_quantile,
    group_trace_stat=group_trace_stat,
    exclude_top_percent_cells=exclude_top_percent_cells,
    paper_axes=paper_axes,
    paper_zoom_per_row=paper_zoom_per_row,
    paper_zoom_quantile=paper_zoom_quantile,
    paper_row_height=paper_row_height,
    figure_width=figure_width,
    trace_linewidth=trace_linewidth,
    overlay_groups_single_axis=overlay_groups_single_axis,
    overlay_alpha=overlay_alpha,
    paper_scalebar_time_value=paper_scalebar_time_value,
    paper_scalebar_time_unit=paper_scalebar_time_unit,
    paper_scalebar_amp_value=paper_scalebar_amp_value,
    plot_inline=plot_inline,
 )

print('trace_linewidth used:', avg_pop_result['trace_linewidth'])
print('overlay enabled:', avg_pop_result['overlay_groups_single_axis'])
print('overlay alpha used:', avg_pop_result['overlay_alpha'])
overlay_files = [p for p in avg_pop_result['saved_files'] if 'population_activity_overlay__' in p]
print('overlay files saved:', len(overlay_files))
for p in overlay_files:
    print(p)

### 41. Cell-averaged deltaF/F

`caban/sections.py` run_avg_population_activity(ds, cfg) with `signal_source='C_dff'`

In [ ]:
session_l = ['TFC_cond', 'Test_B', 'Test_B_1wk']
normalize_mode = None  # None | 'per' | 'across'
same_y_across = True
robust_ylim_quantile = None  # 0.995  # None disables robust clipping
group_trace_stat = 'mean'  # 'mean' | 'median'
exclude_top_percent_cells = None  # e.g., 5 to remove top 5% most active cells

signal_source = 'C_dff'  # 'S_raw' | 'C_dff'
dff_baseline_method = 'rolling_percentile'  # 'rolling_percentile' | 'session_percentile' | 'pre_tone'
dff_baseline_percentile = 20.0
dff_rolling_window_s = 60.0
dff_eps = 1e-6

paper_axes = True
paper_zoom_per_row = True
paper_zoom_quantile = None  # 0.995  # set None to use full extrema
paper_row_height = 0.85  # smaller -> tighter vertical stacking
figure_width = 8.0  # width of figures in inches; adjust for aspect ratio
trace_linewidth = 0.5  # line width for all traces
overlay_groups_single_axis = True  # plot hM3D/hM4D/mCherry on one axis
overlay_alpha = 0.65  # transparency for overlay lines
paper_scalebar_time_value = 5  # e.g., 1, 5
paper_scalebar_time_unit = 'min'  # 'min' | 's' | 'frames'
paper_scalebar_amp_value = 0.5
plot_inline = True  # set True to display figures in the notebook

avg_dff_result = run_avg_population_activity(
    ds,
    cfg,
    sessions=session_l,
    normalize=normalize_mode,
    signal_source=signal_source,
    dff_baseline_method=dff_baseline_method,
    dff_baseline_percentile=dff_baseline_percentile,
    dff_rolling_window_s=dff_rolling_window_s,
    dff_eps=dff_eps,
    same_y_across=same_y_across,
    robust_ylim_quantile=robust_ylim_quantile,
    group_trace_stat=group_trace_stat,
    exclude_top_percent_cells=exclude_top_percent_cells,
    paper_axes=paper_axes,
    paper_zoom_per_row=paper_zoom_per_row,
    paper_zoom_quantile=paper_zoom_quantile,
    paper_row_height=paper_row_height,
    figure_width=figure_width,
    trace_linewidth=trace_linewidth,
    overlay_groups_single_axis=overlay_groups_single_axis,
    overlay_alpha=overlay_alpha,
    paper_scalebar_time_value=paper_scalebar_time_value,
    paper_scalebar_time_unit=paper_scalebar_time_unit,
    paper_scalebar_amp_value=paper_scalebar_amp_value,
    plot_inline=plot_inline,
 )

print('signal used:', avg_dff_result['signal_source'])
print('dF/F baseline method:', avg_dff_result['dff_baseline_method'])
print('dF/F percentile:', avg_dff_result['dff_baseline_percentile'])
print('dF/F rolling window (s):', avg_dff_result['dff_rolling_window_s'])
print('dF/F eps:', avg_dff_result['dff_eps'])
print('baseline tag:', avg_dff_result['baseline_tag'])

In [ ]:
import importlib
import caban.analysis
import caban.sections

importlib.reload(caban.analysis)
importlib.reload(caban.sections)

from caban.sections import run_avg_population_activity


# Rastermap sections

Here we will use the fantastic Rastermap tool to visualize neural activities.
https://github.com/MouseLand/rastermap/tree/main

### 1. Load Rastermap

Using github and the tutorial as a guide.

In [ ]:
import importlib
from IPython.display import display

import caban.sections as caban_sections
importlib.reload(caban_sections)
from caban.sections import run_rastermap_single_mouse

m = 'G05'
session_name = 'Test_B_1wk'

# Rastermap hyperparameters for quick prototyping.
n_PCs = 128 # 64
n_clusters = 100 # 100
locality = 1.0 # 0.75
time_lag_window = None # 5
grid_upsample = None
vmin = 0
vmax = 1.5

rastermap_result = run_rastermap_single_mouse(
    ds,
    m=m,
    session_name=session_name,
    n_PCs=n_PCs,
    n_clusters=n_clusters,
    locality=locality,
    time_lag_window=time_lag_window,
    grid_upsample=grid_upsample,
    vmin=vmin,
    vmax=vmax,
)

### 2. Rastermap parameter sweeps

Used to plot rastermaps for all mice given certain parameter sets, as defined in the previous section.

In [ ]:
import importlib
import gc
import os
import psutil
import matplotlib.pyplot as plt

import caban.sections as caban_sections
importlib.reload(caban_sections)
from caban.sections import run_rastermap_single_mouse

session_l = ['TFC_cond', 'Test_B', 'Test_B_1wk']

# Optional: limit to specific mice for quick inline previews.
# Set to None to run all mice available in each session.
preview_mice = None  # e.g. ['G05']

# Rastermap hyperparameter grids.
# Set any list to None to use a single pass with Rastermap's default for that param.
n_PCs_l = [128]  # [3, 64] # [3, 10, 20, 64]
n_clusters_l = [100]  # [5, 10, 50, 100]
locality_l = [0, 0.5, 1.0]  # [0, 0.25, 0.5, 0.75, 1.0]
time_lag_window_l = [None]  # [0, 5, 10, 20, 40]
grid_upsample = None  # scalar (None -> Rastermap default)
vmin = 0
vmax = 1.5

_n_PCs_l = n_PCs_l if n_PCs_l is not None else [None]
_n_clusters_l = n_clusters_l if n_clusters_l is not None else [None]
_locality_l = locality_l if locality_l is not None else [None]
_time_lag_window_l = time_lag_window_l if time_lag_window_l is not None else [None]

for n_PCs in _n_PCs_l:
    for n_clusters in _n_clusters_l:
        for locality in _locality_l:
            for time_lag_window in _time_lag_window_l:
                param_str = (
                    f"n_PCs={n_PCs}, n_clusters={n_clusters}, locality={locality}, "
                    f"time_lag_window={time_lag_window}, grid_upsample={grid_upsample}"
                )
                print(f"\n[{param_str}]")

                for session_name in session_l:
                    session_dict = getattr(ds, session_name)
                    mice = sorted(session_dict.keys())
                    if preview_mice is not None:
                        mice = [m for m in mice if m in preview_mice]

                    if not mice:
                        raise RuntimeError(f"No mice selected for session {session_name}.")

                    print(f"-> [{session_name}]", end=' ', flush=True)
                    for m in mice:
                        rastermap_result = run_rastermap_single_mouse(
                            ds,
                            m=m,
                            session_name=session_name,
                            n_PCs=n_PCs,
                            n_clusters=n_clusters,
                            locality=locality,
                            time_lag_window=time_lag_window,
                            grid_upsample=grid_upsample,
                            vmin=vmin,
                            vmax=vmax,
                            show_plot=True,  # inline display only; no file saving
                        )

                        # Cleanup after each inline plot.
                        plt.close(rastermap_result['figure'])
                        del rastermap_result
                        gc.collect()
                        print(f"{m}..", end='', flush=True)

                    print("done.", flush=True)

                print(f"Memory usage: {psutil.Process(os.getpid()).memory_info().rss / 1e9:.2f} GB")

# Legacy Analysis sections

These are not yet folded into the refactored jupyter pipeline and/or not fully fleshed out.

### 39. Population vector distances

`caban/main.py` L5938–6367


In [ ]:
run_population_vector_distances(ds, cfg)


### 40. Binned activities

`caban/main.py` L6368–6371


In [ ]:
run_binned_activities(ds, cfg)


# Reloading code after edits

`%autoreload 2` (cell 1) handles most edits. If you re-bind names with `from X import Y`, run this cell to refresh them too.
Reload **bottom-up** (dependencies first).

In [ ]:
import importlib
import caban.utilities, caban.sessions, caban.analysis
import caban.engram, caban.engram_sanity
import caban.population, caban.isomap, caban.epoch_analysis, caban.spatial
import caban.decoder
import caban.config, caban.loader, caban.pipeline
for _m in (caban.utilities, caban.sessions, caban.analysis,
           caban.engram, caban.engram_sanity,
           caban.population, caban.isomap, caban.epoch_analysis, caban.spatial,
           caban.decoder,
           caban.config, caban.loader, caban.pipeline):
    importlib.reload(_m)

# Re-import names that were rebound via `from X import Y`:
from caban.config import PipelineConfig
from caban.loader import load_all_mice
from caban.pipeline import (
    resolve_continuity_params, engram_idx_by_mouse, run_engram_sanity_plots,
    run_population_pca, run_population_pca_all_modes,
    run_isomap, run_epoch_pv, run_cross_session_epoch_pv,
)
print('reload complete')